# Analyse movement bouts
The goal here is to check the shape of the data, specifically if we're able to run the analysis that we want.
## Aim
En substance, we would like to:
* Extract pre-motor activity in moments that the animal is otherwise (relatively) immobile, both in tracking and playback
* Guess features of movement from this activity, such as direction, speed, distance to final position, etc.
* This could be done via dimensionality reduction, direct regression, other sophisticated tech.
## Limits
The list of answerable questions is large, but we need to start by figuring out if the data is plentyful enough

## Parameters
We will select:

**Immobility**

Absolute position velocity stays continuously below 0.02 position units/s for at least 1.2 s.

**Movement onset**

The first sample following a qualifying immobility period. We apply the same definition in tracking and playback.

**Movement endpoint and final position**

Provisionally, the endpoint is the start of the next qualifying immobility period, and final position is Position at that sample. If none follows before the session ends, the endpoint stays missing and the onset remains available for pre-motion analysis.

## Saved info:
We record:
* Identification data: animal, headstage, source session
* Condition: i.e., playback or tracking (0/1, as is already in feature). Set at movement onset (to avoid complicated mixed case)
* onset bin/time
* end bin/time
* Initial position
* Final position
* Total movement
* Direction (initial, to avoid def. problems if it changes, so what matters is the first steps. might be redefined later)
* Speed (avg speed, I guess?)


# 1. Load data

In [ ]:
from pathlib import Path
import pickle

import pandas as pd
from IPython.display import display

# Locate the shared data folder from the workspace, repo, or notebooks folder.
data_dir = next(
    (parent / "data" for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data").is_dir()),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Cannot locate the workspace data directory.")

feature_paths = sorted(data_dir.glob("*_AC_playback_0.005_feature_ss"))
if not feature_paths:
    raise FileNotFoundError(f"No AC feature files found in {data_dir}")

dt = 0.005  # seconds per bin
features = []
session_rows = []
for path in feature_paths:
    animal, _, headstage, region, *_ = path.name.split("_")
    with path.open("rb") as stream:
        collection_features = pickle.load(stream)

    for session, session_features in enumerate(collection_features):
        features.append(session_features)
        session_rows.append({
            "animal": animal,
            "headstage": int(headstage),
            "region": region,
            "session": session,
            "feature_file": path.name,
            "n_bins": len(session_features),
            "duration_s": len(session_features) * dt,
            "tracking_s": session_features["Condition"].eq(0).sum() * dt,
            "playback_s": session_features["Condition"].eq(1).sum() * dt,
        })
    del collection_features, session_features

# features[s][column] corresponds to session_info.loc[s].
# Session entries count each headstage separately.
session_info = pd.DataFrame(session_rows)
print(f"Loaded {len(features)} AC session entries from {len(feature_paths)} collections.")
display(session_info)


# 2. Do itttt

In [ ]:
# Enter parameters for analysis
DT = 0.005
# Speed_x uses position differences * 100. For 5 ms bins, * 2 converts to position units/s.
SPEED_X_FACTOR = 2.0

VELOCITY_THRESHOLD = 0.02
MIN_IMMOBILITY_S = 1.2

In [ ]:
# let's work here while we design a helper function. For now we'll just make the code work on one specific session

idx = 0
# we access features of a specific session by direct ref indexing. So for example: features[idx]["Position"]

import numpy as np
import matplotlib.pyplot as plt

idx = 0

position = features[idx]["Position"].to_numpy()
time = np.arange(len(position)) * DT
speed = features[idx]["Speed_x"].to_numpy() * SPEED_X_FACTOR # fix speed value

### PART ONE: define valid immobility bouts. Every bin will be defined as immobile or not from its instantaneous speed.

# threshold absolute speed
is_immobile_instantaneous = np.abs(speed) < VELOCITY_THRESHOLD

# only keep where there are enough consecutive bins:
changes = np.diff(np.pad(is_immobile_instantaneous.astype(int), (1, 1)))
starts = np.flatnonzero(changes == 1)
stops = np.flatnonzero(changes == -1)
lengths = stops - starts

# select all immobility bouts where length qualifies as sufficiently long
qualifying = lengths * DT >= MIN_IMMOBILITY_S

# store qualifying bouts in a pd.df:
immobility_bouts = pd.DataFrame({
    "start_bin": starts,
    "stop_bin": stops,
    "duration_s": (stops - starts) * DT,
})

immobility_bouts["qualifies"] = (
    immobility_bouts["duration_s"] >= MIN_IMMOBILITY_S
)



### Part two: quantify movement bouts following eligble immobility bouts, and store them somewhere nice and cosy
quiet_bouts = immobility_bouts.loc[immobility_bouts["qualifies"]]

movement_starts = quiet_bouts["stop_bin"].to_numpy()
movement_ends = np.full(len(movement_starts), np.nan)
movement_ends[:-1] = quiet_bouts["start_bin"].to_numpy()[1:]

# An immobility period reaching the session end has no following movement so exclude it
has_onset = movement_starts < len(position)
movement_starts = movement_starts[has_onset]
movement_ends = movement_ends[has_onset]

# One row per movement onset, including bouts without a known endpoint.
info = session_info.loc[idx]
movement_bouts = pd.DataFrame({
    "animal": info["animal"],
    "headstage": info["headstage"],
    "session": info["session"],
    "condition": features[idx]["Condition"].iloc[movement_starts].to_numpy(),
    "onset_bin": movement_starts,
    "onset_time_s": movement_starts * DT,
    "end_bin": pd.array(movement_ends, dtype="Int64"),
    "end_time_s": movement_ends * DT,
    "initial_position": position[movement_starts],
    "initial_direction": np.sign(
        position[movement_starts] - position[movement_starts - 1]
    ),
})

# Endpoint-dependent measurements stay missing for unfinished bouts.
movement_bouts["final_position"] = np.nan
movement_bouts["total_movement"] = np.nan
movement_bouts["mean_speed"] = np.nan

for bout_idx, (onset, end) in enumerate(zip(movement_starts, movement_ends)):
    if np.isfinite(end):
        end = int(end)
        movement_bouts.loc[bout_idx, "final_position"] = position[end]
        # Total distance travelled includes reversals and the endpoint sample.
        movement_bouts.loc[bout_idx, "total_movement"] = np.abs(
            np.diff(position[onset:end + 1])
        ).sum()
        movement_bouts.loc[bout_idx, "mean_speed"] = speed[onset:end].mean()

# Condition is taken at onset, including when a bout crosses a condition boundary.
display(movement_bouts)


In [ ]:
# Browse the full session with a moving time window.
import ipywidgets as widgets
from IPython.display import display
from matplotlib.patches import Patch

window_s = 100  # Width of the visible window, in seconds.
session_duration_s = len(position) * DT
max_start_s = max(0.0, session_duration_s - window_s)

start_slider = widgets.FloatSlider(
    value=min(1000.0, max_start_s),
    min=0.0,
    max=max_start_s,
    step=DT,
    description="Start (s)",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)


def plot_immobility_window(start_s):
    stop_s = min(start_s + window_s, session_duration_s)
    visible = (time >= start_s) & (time < stop_s)
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(time[visible], position[visible], color="slategray", linewidth=1)
    axes[0].set_ylabel("Position")
    axes[1].plot(time[visible], speed[visible], color="slategray", linewidth=1)
    axes[1].axhline(VELOCITY_THRESHOLD, color="purple", linestyle="--",
                    label="Immobility speed threshold")
    axes[1].set_ylabel("Speed (position units/s)")
    axes[1].set_xlabel("Time (s)")

    for bout in immobility_bouts.itertuples(index=False):
        bout_start_s = bout.start_bin * DT
        bout_stop_s = bout.stop_bin * DT  # stop_bin is excluded
        if bout_start_s < stop_s and bout_stop_s > start_s:
            color = "tab:blue" if bout.qualifies else "tab:orange"
            for ax in axes:
                ax.axvspan(max(bout_start_s, start_s), min(bout_stop_s, stop_s),
                           color=color, alpha=0.25, linewidth=0)

    axes[0].legend(handles=[
        Patch(facecolor="tab:blue", alpha=0.25,
              label=f"Immobile for at least {MIN_IMMOBILITY_S:g} s"),
        Patch(facecolor="tab:orange", alpha=0.25, label="Short immobility"),
    ], loc="upper right")
    axes[1].legend(loc="upper right")
    axes[1].set_xlim(start_s, stop_s)
    fig.suptitle(f"Session index {idx}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


bout_plot = widgets.interactive_output(
    plot_immobility_window, {"start_s": start_slider}
)
display(start_slider, bout_plot)
